In [0]:
-- Create permanent table with cleaned and transformed data
CREATE OR REPLACE TABLE workspace.bright_cars.car_sales_clean AS
WITH 
-- Step 1: Remove Duplicates
deduplicated_data AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY vin, saledate 
      ORDER BY 
        CASE WHEN make IS NOT NULL THEN 1 ELSE 0 END DESC,
        CASE WHEN model IS NOT NULL THEN 1 ELSE 0 END DESC,
        CASE WHEN transmission IS NOT NULL THEN 1 ELSE 0 END DESC,
        CASE WHEN odometer > 1 THEN 1 ELSE 0 END DESC,
        odometer DESC,
        condition DESC,
        sellingprice DESC
    ) as row_num
  FROM car_sales_data
),
clean_data AS (
  SELECT * FROM deduplicated_data WHERE row_num = 1
),
-- Step 2: Replace NULL values with 'Unknown' for string columns
null_handled_data AS (
  SELECT
    year,
    COALESCE(make, 'Unknown') as make,
    COALESCE(model, 'Unknown') as model,
    COALESCE(trim, 'Unknown') as trim,
    COALESCE(body, 'Unknown') as body,
    COALESCE(transmission, 'Unknown') as transmission,
    COALESCE(vin, 'Unknown') as vin,
    COALESCE(state, 'Unknown') as state,
    condition,
    odometer,
    COALESCE(color, 'Unknown') as color,
    COALESCE(interior, 'Unknown') as interior,
    COALESCE(seller, 'Unknown') as seller,
    mmr,
    sellingprice,
    saledate
  FROM 
    clean_data
),
-- Step 3: All transformations
transformed_data AS (
  SELECT
    upper(vin) as VIN,
    initcap(concat(make,' ',model)) as Make_Model,
    initcap(model) as Model,
    initcap(trim) as Trim,
    initcap(body) as Body,
    initcap(transmission) as Transmission,
    initcap(interior) as interior,
    initcap(color) as exterior,
    concat(interior,' / ',exterior) as color_combinition, 
    initcap(seller) as Seller,
    initcap(make) as Make,

    -- Manufacture information
    year as year_of_manufacture,
    CASE WHEN year_of_manufacture < 2000 THEN 'Pre 2000' ELSE 'Post 2000' END as Manufacture_Year_Category,

    CASE -- Classify Region of Manufature: What effect does this have on the Revenue? 
      WHEN lower(make) IN ('ford', 'ford tk','chevrolet', 'jeep', 'cadillac', 'gmc', 'buick', 'chrysler', 'dodge', 'ram', 'lincoln', 'tesla') THEN 'American'
      WHEN lower(make) IN ('audi', 'bmw', 'mercedes-b', 'mercedes','mercedes-benz', 'volkswagen', 'porsche', 'mini', 'smart', 'opel', 'peugeot', 'renault', 'fiat', 'alfa romeo', 'citroen', 'volvo', 'saab') THEN 'European'
      WHEN lower(make) IN ('toyota', 'honda', 'nissan', 'mazda', 'subaru', 'mitsubishi', 'lexus', 'acura', 'infiniti', 'suzuki', 'isuzu', 'daihatsu') THEN 'Japanese'
      WHEN lower(make) IN ('hyundai', 'kia', 'genesis', 'ssangyong') THEN 'Korean'
      ELSE 'Other'
    END AS region_of_manufacture,


    -- Date and Time transformations
    to_date(SUBSTR(saledate, 5, 11), 'MMM dd yyyy') as date_sold,
    Extract(DAY FROM date_sold) AS day_of_month,
    date_format(date_sold, 'MMM') AS month_name,
    date_format(date_sold, 'EEE') AS weekday_name,
    Extract(YEAR FROM date_sold) as year_sold,
    
    year_sold - year_of_manufacture as age_at_sale,

    -- Mileage Classifications 
    odometer,
    CASE 
      WHEN odometer <= 20 THEN 'New Car' 
      ELSE 'Used Car' 
      END as New_or_Used,
    CASE
      WHEN odometer > 20 AND odometer <= 150000 THEN 'Low Mileage (< 150k)'
      WHEN odometer > 150000 AND odometer <= 250000 THEN 'Moderate Mileage (150k-250k)'
      ELSE 'High Mileage(>250k)'
    END as mileage_classification,

    -- Condition Classifications
    condition,
    CASE
      WHEN condition IN (1,10) THEN 'very bad'
      WHEN condition BETWEEN 11 AND 20 THEN 'poor'
      WHEN condition BETWEEN 21 AND 30 THEN 'fair'
      WHEN condition BETWEEN 31 AND 40 THEN 'good'
      ELSE 'excellent'
    END As condition_buckets,
    
    -- Calculate Average Yearly Mileage and Classify Based on Average Yearly Mileage
    odometer / NULLIF(age_at_sale, 0) AS average_yearly_mileage,
    CASE
      WHEN average_yearly_mileage < 10000 THEN 'Low Useage'
      WHEN average_yearly_mileage between 10000 and 15000 THEN 'Normal Useage'
      WHEN average_yearly_mileage between 15001 and 20000  THEN 'High Useage'
      WHEN average_yearly_mileage between 20001 and 30000  THEN 'Heavy Wear'
      ELSE 'Abused / Extreme'
    END As yearly_mileage_category,

    -- Pricing and Revenue Calculations
    mmr as cost_price,
    sellingprice as sell_price,
    sellingprice - mmr as front_end_gross_profit,
    Round(100*(sellingprice - mmr)/sellingprice, 2) as Front_End_Gross_Profit_Margin,
CASE
    WHEN Front_End_Gross_Profit_Margin <= 0 THEN 'Loss'
    WHEN New_or_Used = 'New Car' AND Front_End_Gross_Profit_Margin > 0 AND Front_End_Gross_Profit_Margin <= 2 THEN 'Poor'
    WHEN New_or_Used = 'New Car' AND Front_End_Gross_Profit_Margin > 3 AND Front_End_Gross_Profit_Margin <= 5 THEN 'Good'
    WHEN New_or_Used = 'New Car' AND Front_End_Gross_Profit_Margin > 5 AND Front_End_Gross_Profit_Margin <= 8 THEN 'high'
    WHEN New_or_Used = 'New Car' AND Front_End_Gross_Profit_Margin > 8 THEN 'Very high'
    WHEN New_or_Used = 'Used Car' AND Front_End_Gross_Profit_Margin > 0 AND Front_End_Gross_Profit_Margin <= 7 THEN 'Poor'
    WHEN New_or_Used = 'Used Car' AND Front_End_Gross_Profit_Margin > 8 AND Front_End_Gross_Profit_Margin <= 12 THEN 'high'
    WHEN New_or_Used = 'Used Car' AND Front_End_Gross_Profit_Margin > 13 AND Front_End_Gross_Profit_Margin <= 17 THEN 'high'
    WHEN New_or_Used = 'Used Car' AND Front_End_Gross_Profit_Margin > 20 THEN 'Very high'
  END As Profit_Margin_Classification,

    CASE
      WHEN lower(state) = 'al' THEN 'Alabama' 
      WHEN lower(state) = 'ak' THEN 'Alaska' 
      WHEN lower(state) = 'az' THEN 'Arizona'  
      WHEN lower(state) = 'ar' THEN 'Arkansas' 
      WHEN lower(state) = 'ca' THEN 'California' 
      WHEN lower(state) = 'co' THEN 'Colorado' 
      WHEN lower(state) = 'ct' THEN 'Connecticut'  
      WHEN lower(state) = 'de' THEN 'Delaware'  
      WHEN lower(state) = 'dc' THEN 'District of Columbia' 
      WHEN lower(state) = 'fl' THEN 'Florida' 
      WHEN lower(state) = 'ga' THEN 'Georgia' 
      WHEN lower(state) = 'hi' THEN 'Hawaii' 
      WHEN lower(state) = 'id' THEN 'Idaho'  
      WHEN lower(state) = 'il' THEN 'Illinois' 
      WHEN lower(state) = 'in' THEN 'Indiana'  
      WHEN lower(state) = 'ia' THEN 'Iowa'  
      WHEN lower(state) = 'ks' THEN 'Kansas' 
      WHEN lower(state) = 'ky' THEN 'Kentucky' 
      WHEN lower(state) = 'la' THEN 'Louisiana' 
      WHEN lower(state) = 'me' THEN 'Maine' 
      WHEN lower(state) = 'md' THEN 'Maryland' 
      WHEN lower(state) = 'ma' THEN 'Massachusetts' 
      WHEN lower(state) = 'mi' THEN 'Michigan' 
      WHEN lower(state) = 'mn' THEN 'Minnesota' 
      WHEN lower(state) = 'ms' THEN 'Mississippi' 
      WHEN lower(state) = 'mo' THEN 'Missouri' 
      WHEN lower(state) = 'mt' THEN 'Montana' 
      WHEN lower(state) = 'ne' THEN 'Nebraska' 
      WHEN lower(state) = 'nv' THEN 'Nevada' 
      WHEN lower(state) = 'nh' THEN 'New Hampshire' 
      WHEN lower(state) = 'nj' THEN 'New Jersey' 
      WHEN lower(state) = 'nm' THEN 'New Mexico' 
      WHEN lower(state) = 'ny' THEN 'New York' 
      WHEN lower(state) = 'nc' THEN 'North Carolina' 
      WHEN lower(state) = 'nd' THEN 'North Dakota' 
      WHEN lower(state) = 'oh' THEN 'Ohio'  
      WHEN lower(state) = 'ok' THEN 'Oklahoma'  
      WHEN lower(state) = 'or' THEN 'Oregon' 
      WHEN lower(state) = 'pa' THEN 'Pennsylvania' 
      WHEN lower(state) = 'ri' THEN 'Rhode Island' 
      WHEN lower(state) = 'sc' THEN 'South Carolina' 
      WHEN lower(state) = 'sd' THEN 'South Dakota'  
      WHEN lower(state) = 'tn' THEN 'Tennessee'  
      WHEN lower(state) = 'tx' THEN 'Texas'  
      WHEN lower(state) = 'ut' THEN 'Utah' 
      WHEN lower(state) = 'vt' THEN 'Vermont'
      WHEN lower(state) = 'va' THEN 'Virginia' 
      WHEN lower(state) = 'wa' THEN 'Washington'  
      WHEN lower(state) = 'wv' THEN 'West Virginia'
      WHEN lower(state) = 'wi' THEN 'Wisconsin' 
      WHEN lower(state) = 'wy' THEN 'Wyoming'
      WHEN lower(state) = 'unknown' THEN 'Unknown'
      ELSE 'Other'
    END as US_State
  FROM null_handled_data
)
SELECT
  vin,
  make_model,
  make,
  model,
  trim,
  body,
  transmission,
  interior,
  exterior,
  color_combinition,
  year_of_manufacture,
  Manufacture_Year_Category,
  date_sold,
  day_of_month,
  month_name,
  weekday_name,
  year_sold,
  age_at_sale,
  condition,
  condition_buckets,
  average_yearly_mileage,
  yearly_mileage_category,
  mileage_classification,
  sell_price,
  cost_price,
  front_end_gross_profit,
  Front_End_Gross_Profit_Margin,
  Profit_Margin_Classification,
  Seller,
  US_State,
  region_of_manufacture
FROM transformed_data
ORDER BY date_sold DESC, VIN;


select * from workspace.bright_cars.car_sales_clean;


------------------------------------------------------------------------------
-- OBJECTIVE 1: OPTIMIZING INVENTORY
------------------------------------------------------------------------------
-- 1. Pricing Strategy Analysis
-- Identify color combinations that command premiums
-- Insight: Target inventory of Black/Black combos ($17,249 avg) – highest volume AND premium pricing

SELECT color_combinition, COUNT(*) as volume, -- volumne = Units 
       AVG(sell_price) as avg_price,
       AVG(Front_End_Gross_Profit_Margin) as avg_margin
from car_sales_clean
group by color_combinition
HAVING volume > 1000
order by color_combinition desc;



-- 2. Luxury vs Mass-Market Segmentation
-- Segment by interior luxury indicator
-- Insight: Brown/Red interiors = luxury market signal. Use for premium vehicle identification and targeted marketing
SELECT Interior, COUNT(*) as volume,
       AVG(sell_price) as avg_price,
       AVG(Front_End_Gross_Profit_Margin) as avg_margin        
       from car_sales_clean
--WHERE Interior IN ('Brown', 'Red', 'Off-white');
group by Interior;

 -- Luxury interiors

SELECT Interior, exterior, COUNT(*) as volume,
       AVG(sell_price) as avg_price,
       AVG(Front_End_Gross_Profit_Margin) as avg_margin
       from car_sales_clean
WHERE Interior IN ('Black', 'Gray'); -- Standard

-----------------------------------------------------------
-- 2 Best-Selling Color Combinations OVERAL
-- Identify top 30 interior/exterior color combinations by sales volume
-- Shows total units sold, revenue generated, average price, and profitability
SELECT 
  Interior,
  exterior,
  COUNT(*) as total_units_sold,
  SUM(sell_price) as total_revenue,
  AVG(sell_price) as avg_price,
  AVG(Front_End_Gross_Profit_Margin) as avg_margin_pct,
  SUM(front_end_gross_profit) as total_profit,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as pct_of_total_sales
FROM car_sales_clean
WHERE Interior != 'Unknown' AND exterior != 'Unknown'
GROUP BY Interior, exterior
ORDER BY total_units_sold DESC
LIMIT 30;


/*Key Insights:
Black interior dominates: The top 4 combinations all feature black interiors, representing 32.33% of total sales (180,407 units). Black interiors also command higher average prices ($13,718 - $17,250).

Monochromatic combinations lead: Black/Black (#1), Gray/Gray (#7), and Gray/Silver (#6) are extremely popular, suggesting customers prefer coordinated looks.

Gray + neutral exteriors: Gray interiors paired with White, Silver, Gray, or Black represent another 22.79% of sales (127,182 units), making Gray the second most critical interior choice.

Top 10 = 60% of market: The top 10 combinations account for approximately 60.09% of all sales, indicating heavy concentration in neutral color preferences.

Actionable Recommendations:
Stock heavily: Prioritize acquiring Black interior vehicles with Black/Gray/Silver/White exteriors
Premium positioning: Black/Black commands the highest average price ($17,250) – market these as premium options
Avoid exotic combos: Beige/Gold, Tan/Gold, and rare colors represent <1% each – minimize inventory
Bundle strategy: Since customers prefer neutrals, create bundles or promotions around "classic combinations"

*/

-- Major Body Types - Top Color Combinations:

SELECT 
  body,
  color_combinition,
  COUNT(*) as total_units_sold,
  SUM(sell_price) as total_revenue,
  AVG(sell_price) as avg_price,
  AVG(Front_End_Gross_Profit_Margin) as avg_margin_pct,
  SUM(front_end_gross_profit) as total_profit,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as pct_of_total_sales
FROM car_sales_clean
--WHERE Interior != 'Unknown' AND exterior != 'Unknown'
GROUP BY body, color_combinition
ORDER BY total_units_sold DESC;


/*
Sedans (Largest segment: 27,047 units in #1 combo)
Black/Black: 27,047 units | $14,870 avg
Black/Gray: 19,756 units | $13,435 avg
Black/Silver: 19,611 units | $11,917 avg
Gray/Silver: 16,855 units | $8,555 avg
Black/White: 16,216 units | $14,438 avg
Pattern: Black interiors dominate; strong preference for monochromatic or neutral combos.

SUVs (Premium pricing)
Black/Black: 17,289 units | $20,578 avg (+38% vs Sedans)
Black/Silver: 10,513 units | $17,361 avg
Black/Gray: 10,466 units | $18,926 avg
Gray/Silver: 8,404 units | $11,368 avg
Black/White: 8,206 units | $20,855 avg
Pattern: Black interiors command significantly higher prices in SUVs; customers willing to pay premium for black.

Trucks (Crew Cab, Supercrew, Double Cab)
Crew Cab:

Gray/White: 1,918 units | $21,308 avg
Black/White: 1,781 units | $25,891 avg
Black/Black: 1,684 units | $24,770 avg
Supercrew:

Gray/White: 1,083 units | $20,339 avg
Black/Black: 744 units | $26,244 avg
Pattern: Gray interiors dominate trucks by volume, but Black interiors sell for +20-30% higher prices. Work trucks = Gray; Personal trucks = Black.

Minivans (Family vehicles)
Gray/Silver: 2,402 units | $8,137 avg
Gray/White: 2,003 units | $9,382 avg
Gray/Gray: 1,996 units | $12,583 avg
Black/White: 1,954 units | $14,883 avg (+83% vs Gray/Silver!)
Pattern: Gray interiors are standard (easier to clean), but Black commands massive premium for upscale families.

Convertibles (Luxury segment)
Black/Black: 1,395 units | $21,785 avg
Black/Silver: 929 units | $14,220 avg
Black/Red: 708 units | $17,557 avg
Pattern: Black interior is non-negotiable for convertibles (UV protection + luxury appeal). Red exterior is popular accent color.

Hatchbacks (Compact/economy)
Black/Black: 2,777 units | $10,889 avg
Black/White: 2,510 units | $10,948 avg
Black/Gray: 2,405 units | $10,808 avg
Pattern: Even budget buyers prefer Black interiors for style.

Key Business Insights:
Interior Color Strategy by Body Type:
SUVs, Convertibles, Coupes: Black interior is premium differentiator (+$2K-$6K premium)
Trucks: Gray for fleet/work buyers; Black for personal use (25% price premium)
Minivans: Gray for practicality, but Black taps premium family market (+83% price)
Sedans/Hatchbacks: Black is universal preference across all price points
Actionable Recommendations:
Acquisition Strategy:

SUVs: Prioritize Black interior + Black/White/Gray exteriors for maximum margin
Trucks: Stock Gray/White for volume; Black/Black for premium margins
Minivans: Gray/Silver is safe; Black/White for upselling opportunities
Convertibles/Coupes: Only acquire Black interiors (90%+ of market)
Pricing Strategy:

Charge +15-20% premium for Black interiors on SUVs and trucks
In Minivans, Black interior justifies +50% markup over Gray
Sedans: Black/Black is volume leader—price competitively but stock heavily
*/


-----------------------------------------------------------
-- 4. MARGIN ANALYSIS BY COLOR AND BODY TYPE (CORRECTED)
-- Identify most profitable color/body combinations
-- Focus on combinations with volume >100 for statistical significance
-- Uses TOTAL PROFIT DOLLARS as primary ranking metric
WITH margin_analysis AS (
  SELECT 
    Body,
    Interior,
    exterior,
    COUNT(*) as total_units,
    AVG(sell_price) as avg_sell_price,
    AVG(cost_price) as avg_cost_price,
    -- CORRECTED: Revenue-weighted margin (actual margin %)
    (SUM(front_end_gross_profit) / NULLIF(SUM(sell_price), 0)) * 100 as revenue_weighted_margin_pct,
    -- Average margin (for comparison - can be misleading)
    AVG(Front_End_Gross_Profit_Margin) as avg_margin_pct,
    -- PRIMARY METRIC: Total profit in dollars
    SUM(front_end_gross_profit) as total_profit_dollars,
    AVG(front_end_gross_profit) as avg_profit_per_unit,
    SUM(sell_price) as total_revenue,
    -- Categorize margin performance
    CASE 
      WHEN (SUM(front_end_gross_profit) / NULLIF(SUM(sell_price), 0)) * 100 > 5 THEN 'High Margin'
      WHEN (SUM(front_end_gross_profit) / NULLIF(SUM(sell_price), 0)) * 100 BETWEEN 0 AND 5 THEN 'Moderate Margin'
      ELSE 'Loss/Low Margin'
    END as margin_category
  FROM car_sales_clean
  WHERE Interior != 'Unknown' AND exterior != 'Unknown' AND Body != 'Unknown'
  GROUP BY Body, Interior, exterior
  HAVING total_units >= 100  -- Focus on statistically significant volumes
)
SELECT 
  Body,
  Interior,
  exterior,
  total_units,
  avg_sell_price,
  -- Show both margins for comparison
  avg_margin_pct as avg_margin_misleading,
  revenue_weighted_margin_pct as true_margin_pct,
  total_profit_dollars,
  avg_profit_per_unit,
  total_revenue,
  margin_category,
  -- Rank within each body type by total profit (PRIMARY METRIC)
  ROW_NUMBER() OVER (PARTITION BY Body ORDER BY total_profit_dollars DESC) as profit_rank_in_body,
  -- Rank overall by total profit dollars (PRIMARY METRIC)
  ROW_NUMBER() OVER (ORDER BY total_profit_dollars DESC) as profit_rank_overall
FROM margin_analysis
ORDER BY total_profit_dollars DESC
LIMIT 50;